# Ridge Logistic Regression xG Model

Trains and evaluates an L2-penalised (Ridge) logistic regression expected-goals model on StatsBomb open data.

Ridge logistic regression is a natural baseline for xG modelling:
- Outputs well-calibrated probabilities with no extra calibration step
- Coefficients are directly interpretable as feature effects on log-odds
- Regularisation prevents overfitting without requiring tree-specific tuning

**Pipeline summary:**
1. Load data and engineer features (reuses `src.features`)
2. Preprocess — standardise numerics, encode categoricals, impute
3. Tune regularisation strength `C` with Optuna (5-fold CV, log-loss)
4. Final cross-validated evaluation — AUC, log-loss, Brier score
5. Calibration curve
6. Coefficient analysis
7. Distance-only baseline comparison
8. Save model to `outputs/`

In [ ]:
import sys, warnings, os
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import joblib
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, log_loss, brier_score_loss,
    RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.calibration import calibration_curve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from statsbombpy import sb

from src.features import create_xg_features

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
os.makedirs('../outputs', exist_ok=True)

## 1. Data Loading and Feature Engineering

In [ ]:
COMPETITIONS = [
    {'competition_id': 11, 'season_id': 27,  'label': 'La Liga 2015/16'},
    {'competition_id': 16, 'season_id': 4,   'label': 'Champions League 2018/19'},
    {'competition_id': 2,  'season_id': 27,  'label': 'Premier League 2015/16'},
    {'competition_id': 43, 'season_id': 106, 'label': 'World Cup 2022'},
]

def load_competition(competition_id, season_id):
    matches = sb.matches(competition_id=competition_id, season_id=season_id)
    return pd.concat(
        [sb.events(match_id=mid).assign(match_id=mid) for mid in matches['match_id']],
        ignore_index=True,
    )

all_events = []
for comp in COMPETITIONS:
    print(f"Loading {comp['label']}…")
    all_events.append(load_competition(comp['competition_id'], comp['season_id']))

events_df = pd.concat(all_events, ignore_index=True)
xg_df = create_xg_features(events_df)

shots = xg_df[xg_df['is_penalty'] == 0].copy().reset_index(drop=True)

print(f"\nShots (excl. penalties): {len(shots):,}")
print(f"Goals: {shots['is_goal'].sum():,} ({shots['is_goal'].mean():.1%})")

## 2. Preprocessing

Ridge regression is sensitive to feature scale, so numeric features are standardised with `StandardScaler`. This is the key preprocessing difference from the tree-based XGBoost model.

In [ ]:
CATEGORICAL_FEATURES = ['shot_body_part', 'shot_technique', 'previous_event_type']

NUMERIC_FEATURES = [
    'distance_to_goal', 'shot_angle', 'centrality',
    'in_penalty_area', 'in_six_yard_box',
    'shot_first_time', 'shot_under_pressure',
    'num_defenders_in_frame', 'num_teammates_in_frame',
    'distance_to_nearest_defender', 'num_defenders_between_shot_and_goal',
    'goalkeeper_distance_to_goal', 'goalkeeper_distance_to_shooter',
    'previous_event_was_pass', 'previous_event_was_carry',
    'previous_event_same_team', 'previous_event_distance',
    'is_late_game', 'is_extra_time',
]

TARGET = 'is_goal'

numeric_pipeline = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    # Standardisation is required: Ridge penalises large coefficients uniformly,
    # so features on different scales would be penalised unequally without scaling.
    ('scale',  StandardScaler()),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, NUMERIC_FEATURES),
    ('cat', Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='Unknown')),
        ('ohe',    OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), CATEGORICAL_FEATURES),
], remainder='drop')

X = shots[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = shots[TARGET].values

X_transformed = preprocessor.fit_transform(X)
ohe_feature_names = (
    preprocessor.named_transformers_['cat']['ohe']
    .get_feature_names_out(CATEGORICAL_FEATURES)
    .tolist()
)
ALL_FEATURE_NAMES = NUMERIC_FEATURES + ohe_feature_names
print(f"Feature matrix: {X_transformed.shape[0]:,} shots × {X_transformed.shape[1]} features")

## 3. Hyperparameter Tuning (Optuna)

`C` is the inverse regularisation strength — smaller `C` = stronger Ridge penalty.

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    C = trial.suggest_float('C', 1e-4, 1e3, log=True)
    model = Pipeline([
        ('pre', preprocessor),
        ('clf', LogisticRegression(
            penalty='l2', C=C, solver='lbfgs',
            max_iter=1000, random_state=42,
        )),
    ])
    fold_losses = []
    for train_idx, val_idx in CV.split(X, y):
        model.fit(X.iloc[train_idx], y[train_idx])
        proba = model.predict_proba(X.iloc[val_idx])[:, 1]
        fold_losses.append(log_loss(y[val_idx], proba))
    return np.mean(fold_losses)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=40, show_progress_bar=True)

best_C = study.best_params['C']
print(f"\nBest log-loss: {study.best_value:.4f}")
print(f"Best C:        {best_C:.5f}")

In [ ]:
# C search landscape — shows how sensitive performance is to regularisation strength
trials_df = study.trials_dataframe().sort_values('params_C')

fig, ax = plt.subplots(figsize=(8, 3))
ax.semilogx(trials_df['params_C'], trials_df['value'], 'o', alpha=0.6, ms=5, color='#4C72B0')
ax.axvline(best_C, color='#DD8452', linewidth=1.5, linestyle='--', label=f'Best C={best_C:.4f}')
ax.set_xlabel('C (log scale)')
ax.set_ylabel('CV log-loss')
ax.set_title('Regularisation strength search')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Final Cross-Validated Evaluation

In [ ]:
final_model = Pipeline([
    ('pre', preprocessor),
    ('clf', LogisticRegression(
        penalty='l2', C=best_C, solver='lbfgs',
        max_iter=1000, random_state=42,
    )),
])

oof_proba = cross_val_predict(final_model, X, y, cv=CV, method='predict_proba')[:, 1]

metrics = {
    'ROC-AUC':     roc_auc_score(y, oof_proba),
    'Log-loss':    log_loss(y, oof_proba),
    'Brier score': brier_score_loss(y, oof_proba),
}

print("Out-of-fold evaluation metrics:")
for name, val in metrics.items():
    print(f"  {name:<15} {val:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

RocCurveDisplay.from_predictions(y, oof_proba, ax=axes[0], color='#4C72B0')
axes[0].plot([0, 1], [0, 1], 'k--', linewidth=0.8)
axes[0].set_title('ROC curve (out-of-fold)')

PrecisionRecallDisplay.from_predictions(y, oof_proba, ax=axes[1], color='#DD8452')
axes[1].axhline(y.mean(), color='k', linestyle='--', linewidth=0.8, label=f'Baseline ({y.mean():.3f})')
axes[1].legend()
axes[1].set_title('Precision-Recall curve (out-of-fold)')

plt.tight_layout()
plt.show()

## 5. Calibration

In [ ]:
fraction_pos, mean_pred = calibration_curve(y, oof_proba, n_bins=15, strategy='quantile')

fig, ax = plt.subplots(figsize=(6, 5))
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8, label='Perfect calibration')
ax.plot(mean_pred, fraction_pos, 'o-', color='#4C72B0', linewidth=2, markersize=6, label='Ridge xG')
ax.set_xlabel('Mean predicted xG')
ax.set_ylabel('Fraction of goals')
ax.set_title('Calibration curve')
ax.legend()
ax.set_xlim(0, 0.7)
ax.set_ylim(0, 0.7)
plt.tight_layout()
plt.show()

bins = np.linspace(0, 1, 11)
bin_ids = np.digitize(oof_proba, bins) - 1
ece = sum(
    (np.sum(bin_ids == b) / len(y)) * abs(y[bin_ids == b].mean() - oof_proba[bin_ids == b].mean())
    for b in range(len(bins) - 1) if np.sum(bin_ids == b) > 0
)
print(f"Expected Calibration Error (ECE): {ece:.4f}")

## 6. Coefficient Analysis

Unlike tree-based models, logistic regression coefficients are directly interpretable: each coefficient is the change in log-odds of a goal per one-standard-deviation increase in that feature (for numeric features after scaling).

In [ ]:
final_model.fit(X, y)
coefs = pd.Series(
    final_model.named_steps['clf'].coef_[0],
    index=ALL_FEATURE_NAMES,
).sort_values()

# Show top and bottom 15 coefficients
n = 15
coef_display = pd.concat([coefs.head(n), coefs.tail(n)]).drop_duplicates()
colors = ['#DD8452' if v > 0 else '#4C72B0' for v in coef_display]

fig, ax = plt.subplots(figsize=(8, 8))
coef_display.plot.barh(ax=ax, color=colors, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coefficient (log-odds per SD for numeric features)')
ax.set_title(f'Ridge logistic regression — top/bottom {n} coefficients')
plt.tight_layout()
plt.show()

In [ ]:
# Odds ratios — easier to communicate: OR > 1 increases goal probability, OR < 1 decreases it
odds_ratios = np.exp(coefs)
or_display = pd.concat([odds_ratios.head(n), odds_ratios.tail(n)]).drop_duplicates().sort_values()

colors_or = ['#DD8452' if v > 1 else '#4C72B0' for v in or_display]
fig, ax = plt.subplots(figsize=(8, 8))
or_display.plot.barh(ax=ax, color=colors_or, edgecolor='white')
ax.axvline(1, color='black', linewidth=0.8)
ax.set_xlabel('Odds ratio')
ax.set_title(f'Ridge logistic regression — top/bottom {n} odds ratios')
plt.tight_layout()
plt.show()

print("Top 10 goal-increasing features (OR > 1):")
print(odds_ratios.sort_values(ascending=False).head(10).round(3).to_string())
print("\nTop 10 goal-decreasing features (OR < 1):")
print(odds_ratios.sort_values().head(10).round(3).to_string())

## 7. Baseline Comparison

A distance-only logistic regression is the simplest meaningful xG model. Comparing against it shows how much the additional features contribute.

In [ ]:
baseline_model = Pipeline([
    ('pre', Pipeline([
        ('impute', SimpleImputer(strategy='median')),
        ('scale',  StandardScaler()),
    ])),
    ('clf', LogisticRegression(penalty='l2', C=best_C, solver='lbfgs',
                               max_iter=1000, random_state=42)),
])

X_baseline = shots[['distance_to_goal', 'shot_angle']]
baseline_proba = cross_val_predict(
    baseline_model, X_baseline, y, cv=CV, method='predict_proba'
)[:, 1]

baseline_metrics = {
    'ROC-AUC':     roc_auc_score(y, baseline_proba),
    'Log-loss':    log_loss(y, baseline_proba),
    'Brier score': brier_score_loss(y, baseline_proba),
}

comparison = pd.DataFrame({
    'Distance + angle only': baseline_metrics,
    'Ridge (all features)':  metrics,
}).T.round(4)

print("Model comparison (out-of-fold):")
print(comparison.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

RocCurveDisplay.from_predictions(
    y, baseline_proba, ax=ax, color='#4C72B0',
    name=f"Distance + angle  (AUC={baseline_metrics['ROC-AUC']:.3f})",
)
RocCurveDisplay.from_predictions(
    y, oof_proba, ax=ax, color='#DD8452',
    name=f"Ridge all features (AUC={metrics['ROC-AUC']:.3f})",
)
ax.plot([0, 1], [0, 1], 'k--', linewidth=0.8)
ax.set_title('ROC comparison — baseline vs full Ridge model')
plt.tight_layout()
plt.show()

## 8. xG Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for outcome, label, color in [(0, 'No goal', '#4C72B0'), (1, 'Goal', '#DD8452')]:
    axes[0].hist(oof_proba[y == outcome], bins=40, alpha=0.55,
                 label=label, color=color, density=True, edgecolor='none')
axes[0].set_xlabel('Predicted xG')
axes[0].set_ylabel('Density')
axes[0].set_title('xG distribution by outcome')
axes[0].legend()

shots_eval = shots.copy()
shots_eval['xg'] = oof_proba
match_summary = shots_eval.groupby('match_id').agg(
    xG_total=('xg', 'sum'),
    actual_goals=('is_goal', 'sum'),
)
axes[1].scatter(match_summary['xG_total'], match_summary['actual_goals'],
                alpha=0.4, s=20, color='#4C72B0')
lim = max(match_summary[['xG_total', 'actual_goals']].max()) + 0.5
axes[1].plot([0, lim], [0, lim], 'k--', linewidth=0.8)
axes[1].set_xlabel('Sum xG per match')
axes[1].set_ylabel('Actual goals per match')
axes[1].set_title('xG vs goals per match')

plt.tight_layout()
plt.show()

match_corr = match_summary.corr().loc['xG_total', 'actual_goals']
print(f"Match-level Pearson r (xG vs goals): {match_corr:.3f}")

## 9. Save Model

In [ ]:
model_path = '../outputs/ridge_xg_model.pkl'
joblib.dump(final_model, model_path)
print(f"Model saved to {model_path}")

reloaded = joblib.load(model_path)
check = reloaded.predict_proba(X.iloc[:5])[:, 1]
ref   = final_model.predict_proba(X.iloc[:5])[:, 1]
assert np.allclose(check, ref), "Reloaded model predictions differ!"
print("Reload check passed.")

## 10. Results Summary

In [ ]:
print("="*45)
print(" Ridge Logistic Regression xG Model")
print("="*45)
print(f" Dataset:       {len(shots):,} open-play shots")
print(f" Goal rate:     {y.mean():.1%}")
print(f" Features:      {len(ALL_FEATURE_NAMES)} (after encoding)")
print(f" Best C:        {best_C:.5f}")
print(f" CV folds:      5-fold stratified")
print("-"*45)
print(" Full model (all features):")
for name, val in metrics.items():
    print(f"   {name:<15} {val:.4f}")
print(f"   {'ECE':<15} {ece:.4f}")
print("-"*45)
print(" Baseline (distance + angle only):")
for name, val in baseline_metrics.items():
    print(f"   {name:<15} {val:.4f}")
print("-"*45)
print(f" Model saved:   {model_path}")
print("="*45)